# AR Workflow NB 1: Goldenson et al (2018) AR Detection algorithm.
### See below comments for description.

In [1]:
# This AR detection algorithm is modified from Goldenson et al (2018).
# It uses IVT instead of IWV and has functionality for multiple RCMs.
import numpy as np
from scipy.ndimage import label, extrema
from scipy.ndimage import sum as ndSum
from scipy.ndimage import maximum as ndMax
import pandas as pd
import xarray as xy
import netCDF4
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import string
from matplotlib import cm  # colormap
import os.path
from copy import deepcopy
from time import time
# vincenty is no longer supported, geodesic does the same thing.
from geopy.distance import geodesic
from glob import glob
import cftime
import dask
from dask.distributed import LocalCluster, Client

In [2]:
cluster = LocalCluster(n_workers=4, processes=True, memory_limit='64GB')
client = Client(cluster)

In [3]:
class GeographicBounds(object):
    def __init__(self, lonMin, lonMax, latMin, latMax):
        self.lonMin = lonMin
        self.lonMax = lonMax
        self.latMin = latMin
        self.latMax = latMax


# I may not need this, or come up with my own, considering the cold season 
#     of focus is interseasonal (Oct thru March).
def startMonth(season):
    if season == 'DJF':
        return 12
    elif season == 'MAM':
        return 3
    elif season == 'JJA':
        return 6
    elif season == 'SON':
        return 9


# land fraction is the same for all runs on the same mesh:
land_filename = '/glade/derecho/scratch/tcorrie/regrids/landmask_regridded.nc' # Change

toGetLandFrac = xy.open_dataset(land_filename)
# May change but may leave alone and just consider the regions marked below.
bounds = GeographicBounds(-140, -110, 31.5, 50.001)
toGetLandFrac = toGetLandFrac.sel(lon=slice(bounds.lonMin, bounds.lonMax),
                                  lat=slice(bounds.latMin, bounds.latMax))
# error in original code: switch latMin and latMax

landFracArr = np.asarray(toGetLandFrac['LANDMASK'].squeeze())
# -**-data-set dependent-**---
HALFCELLINDEG = 0.05  # Original was 0.3515625... odd


# ----------------------------
def getCellArea(lon, lat):
    y = geodesic((lat-HALFCELLINDEG, 0), (lat+HALFCELLINDEG, 0)).kilometers
    x = geodesic((lat, lon-HALFCELLINDEG), (lat, lon+HALFCELLINDEG)).kilometers
    return x * y


# can also do this part once, since all the time steps are on the same grid:
latArr = np.asarray(toGetLandFrac.lat)
lonArr = np.asarray(toGetLandFrac.lon)
lonArr, latArr = np.meshgrid(lonArr, latArr)
cellAreaArr = deepcopy(lonArr)
for i in range(len(latArr)):
    # Change so this becomes the j dimension. len(---Arr) was
    #     returning the i dimension.
    for j in range(lonArr.shape[1]):
        cellAreaArr[i, j] = getCellArea(lonArr[i, j], latArr[i, j])


def getARmask(tmqOne):
    # tmqOne is whatever the integrated moisture field is. Original is for IWV
    #     but this is IVT.
    maxArr = np.asarray(tmqOne).squeeze()
    THRESHOLD = 250
    maxArr = np.where(maxArr < THRESHOLD, 0, maxArr)
    labels, numL = label(maxArr)
    onlyOneLabel = np.zeros(labels.shape)
    if numL > 0:
        sizesOfLabeledRegions = [len(labels[labels == i]) for i in range(1, numL + 1)]
        whichLabel = np.argmax(sizesOfLabeledRegions)
        onlyOneLabel = np.zeros(labels.shape)
        onlyOneLabel[np.nonzero(labels == whichLabel+1)] = 1

        lats = extrema(latArr, onlyOneLabel)
        lons = extrema(lonArr, onlyOneLabel)
        lowerLeft = (lats[0], lons[0])
        upperRight = (lats[1], lons[1])
        areaSum = ndSum(cellAreaArr, onlyOneLabel)
        lengthKM = geodesic(lowerLeft, upperRight).kilometers
        width = areaSum/lengthKM
        if lengthKM/width >= 2:
            return xy.DataArray(onlyOneLabel, dims=tmqOne.dims, coords=tmqOne.coords, name='ARmask')
        else:
            zerosLabel = np.zeros_like(onlyOneLabel)
            return xy.DataArray(zerosLabel, dims=tmqOne.dims, coords=tmqOne.coords, name='ARmask')
    else:
        zerosLabel = np.zeros_like(onlyOneLabel)
        return xy.DataArray(zerosLabel, dims=tmqOne.dims, coords=tmqOne.coords, name='ARmask')

In [4]:
directoryData = '/glade/derecho/scratch/tcorrie/regrids/'  # Change
outputdir = '/glade/work/tcorrie/ARdata/ARmasks/G18/'
field = 'ivt'

gcm_models = pd.read_csv('../wusd3_gcms.csv', index_col=0)

for i in range(1011, 1201, 20):
    gcm_models.loc[len(gcm_models)] = ['cesm2-le', 'CESM2-LE', str(i), '365-day']

def model_ARmasks(model, member):
    print(model, member, " begin.")
    glob_file = directoryData + f'ivt.daily.{model}.{member}.d01_regridded.nc'
    oneFile = xy.open_dataset(glob_file)
    t1 = time()
    tmq = oneFile.squeeze(dim=['lev','ens'], drop=True)
    tmq[field] = (tmq['ivtx']**2 + tmq['ivty']**2)**0.5
    # Swapped latMin and latMax here as well
    tmq = tmq.sel(lon=slice(bounds.lonMin, bounds.lonMax),
                  lat=slice(bounds.latMin, bounds.latMax))
    tmq = tmq[field]
    armasks = [getARmask(tmq.isel(time=t)) for t in range(tmq.sizes['time'])]
    events = xy.concat(armasks, dim=tmq.time).to_dataset(name='ARmask')
    t2 = time()
    mins = (t2-t1) // 60
    secs = (t2-t1) % 60
    events.to_netcdf(outputdir + f'ARmask.{model}.{member}.nc')
    print(f'finding {name}/{member} ARs took {mins:.0f}m {secs:04.1f}s ')

delayed_tasks = []
for i in range(len(gcm_models)):
    model = gcm_models.iat[i, 0]
    name = gcm_models.iat[i, 1]
    member = gcm_models.iat[i, 2]
    cd = gcm_models.iat[i, 3]

    model_ARmasks(model, member)
    # If you would like to parallelize this process, uncomment the next two lines.
    #delayed_tasks.append(dask.delayed(model_ARmasks)(model, member))
    
#out = dask.compute(*delayed_tasks)


access-cm2 r5i1p1f1  begin.


/glade/derecho/scratch/tcorrie/tmp/ipykernel_108991/310243178.py:76: RuntimeWarning: divide by zero encountered in scalar divide
  width = areaSum/lengthKM


finding ACCESS-CM2/r5i1p1f1 ARs took 1m 58.8s 
canesm5 r1i1p2f1  begin.


/glade/derecho/scratch/tcorrie/tmp/ipykernel_108991/310243178.py:76: RuntimeWarning: divide by zero encountered in scalar divide
  width = areaSum/lengthKM


finding CanESM5/r1i1p2f1 ARs took 1m 46.7s 
cesm2 r11i1p1f1  begin.


...

finding CESM2-LE/1151 ARs took 3m 51.8s 
cesm2-le 1171  begin.


/glade/derecho/scratch/tcorrie/tmp/ipykernel_108991/310243178.py:76: RuntimeWarning: divide by zero encountered in scalar divide
  width = areaSum/lengthKM


finding CESM2-LE/1171 ARs took 3m 48.6s 
cesm2-le 1191  begin.


/glade/derecho/scratch/tcorrie/tmp/ipykernel_108991/310243178.py:76: RuntimeWarning: divide by zero encountered in scalar divide
  width = areaSum/lengthKM


finding CESM2-LE/1191 ARs took 3m 45.2s 


In [ ]:
# Select this cell and run all above to run the AR detection algorithm.